In [ ]:
# V-JEPA2 Action Anticipation Training on EPIC-Kitchens 100
# Using HuggingFace cached dataset + official annotations
# Config: 64 frames per clip, 4 fps

import sys
sys.path.insert(0, '/home/jefferyfan/Archive/IME/vjepa2')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Tuple, Optional, List
import warnings
warnings.filterwarnings('ignore')

# Check device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

## Configuration

Training config adapted from V-JEPA2 paper:
- 64 frames per clip (changed from 32)
- 4 fps (changed from 8)
- anticipation_time_sec: 1.0
- resolution: 256

In [ ]:
# Configuration
CONFIG = {
    # Data paths
    'hf_cache_path': '/home/jefferyfan/.cache/huggingface/hub/datasets--awsaf49--epic_kitchens_100/snapshots/c5e573b9295c06bd6d14bd5a9b9c7b178f00d7b1',
    'train_annotations': '/home/jefferyfan/Archive/IME/EPIC_100_train.csv',
    'val_annotations': '/home/jefferyfan/Archive/IME/EPIC_100_validation.csv',
    
    # Video settings (MODIFIED from original config)
    'frames_per_clip': 64,      # Changed from 32
    'frames_per_second': 4,     # Changed from 8
    'resolution': 256,
    
    # Anticipation settings
    'anticipation_time_sec': (1.0, 1.0),
    'train_anticipation_point': (0.0, 0.25),
    'train_anticipation_time_sec': (0.25, 1.75),
    
    # Model settings
    'num_probe_blocks': 4,
    'num_heads': 16,
    
    # Training settings  
    'batch_size': 2,
    'num_epochs': 20,
    'learning_rate': 0.001,
    'weight_decay': 0.0001,
    'num_workers': 4,
    
    # V-JEPA2 model settings
    'encoder': {
        'model_name': 'vit_large',
        'tubelet_size': 2,
        'patch_size': 16,
        'uniform_power': True,
        'use_rope': True,
    },
    'predictor': {
        'model_name': 'vit_predictor',
        'num_frames': 64,
        'depth': 12,
        'num_heads': 12,
        'predictor_embed_dim': 384,
        'num_mask_tokens': 10,
        'uniform_power': True,
        'use_mask_tokens': True,
        'use_sdpa': True,
        'use_rope': True,
    },
    'wrapper': {
        'no_predictor': False,
        'num_output_frames': 2,
        'num_steps': 1,
    }
}

print("Configuration loaded:")
print(f"  Frames per clip: {CONFIG['frames_per_clip']}")
print(f"  FPS: {CONFIG['frames_per_second']}")
print(f"  Context duration: {CONFIG['frames_per_clip'] / CONFIG['frames_per_second']:.1f} seconds")

## Load Annotations and Build Class Mappings

In [ ]:
# Load annotations
train_df = pd.read_csv(CONFIG['train_annotations'])
val_df = pd.read_csv(CONFIG['val_annotations'])

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"\nColumns: {train_df.columns.tolist()}")
print(f"\nSample annotation:")
print(train_df.iloc[0])

# Build class mappings
all_df = pd.concat([train_df, val_df])

verb_classes = sorted(all_df['verb_class'].unique())
noun_classes = sorted(all_df['noun_class'].unique())

# Action class = verb_class * 1000 + noun_class (for unique action IDs)
all_df['action_class'] = all_df['verb_class'] * 1000 + all_df['noun_class']
action_classes = sorted(all_df['action_class'].unique())

VERB_CLASSES = {v: i for i, v in enumerate(verb_classes)}
NOUN_CLASSES = {n: i for i, n in enumerate(noun_classes)}
ACTION_CLASSES = {a: i for i, a in enumerate(action_classes)}

print(f"\nNumber of verb classes: {len(VERB_CLASSES)}")
print(f"Number of noun classes: {len(NOUN_CLASSES)}")
print(f"Number of action classes: {len(ACTION_CLASSES)}")

## Dataset Class for EPIC-Kitchens 100

In [ ]:
import random
from decord import VideoReader, cpu
import torchvision.transforms as T

class EK100AnticipationDataset(Dataset):
    """EPIC-Kitchens 100 dataset for action anticipation using HF cached videos"""
    
    def __init__(
        self,
        annotations_df: pd.DataFrame,
        video_root: str,
        frames_per_clip: int = 64,
        fps: int = 4,
        resolution: int = 256,
        anticipation_time_sec: Tuple[float, float] = (1.0, 1.0),
        anticipation_point: Tuple[float, float] = (0.0, 0.25),
        verb_classes: Dict = None,
        noun_classes: Dict = None,
        action_classes: Dict = None,
        training: bool = True,
    ):
        self.annotations = annotations_df.reset_index(drop=True)
        self.video_root = Path(video_root)
        self.frames_per_clip = frames_per_clip
        self.fps = fps
        self.resolution = resolution
        self.anticipation_time_sec = anticipation_time_sec
        self.anticipation_point = anticipation_point
        self.verb_classes = verb_classes or VERB_CLASSES
        self.noun_classes = noun_classes or NOUN_CLASSES
        self.action_classes = action_classes or ACTION_CLASSES
        self.training = training
        
        # Video transforms
        self.transform = T.Compose([
            T.ToPILImage(),
            T.Resize((resolution, resolution)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        
        # Build video path lookup
        self._build_video_paths()
        
    def _build_video_paths(self):
        """Build mapping from video_id to video file path"""
        self.video_paths = {}
        for participant_dir in self.video_root.iterdir():
            if participant_dir.is_dir() and participant_dir.name.startswith('P'):
                videos_dir = participant_dir / 'videos'
                if videos_dir.exists():
                    for video_file in videos_dir.glob('*.MP4'):
                        video_id = video_file.stem
                        self.video_paths[video_id] = str(video_file)
        print(f"Found {len(self.video_paths)} videos")
        
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx: int) -> Dict:
        row = self.annotations.iloc[idx]
        
        video_id = row['video_id']
        start_frame = row['start_frame']
        stop_frame = row['stop_frame']
        verb_class = row['verb_class']
        noun_class = row['noun_class']
        
        # Get video path
        video_path = self.video_paths.get(video_id)
        if video_path is None:
            # Return dummy data if video not found
            return self._get_dummy_sample(verb_class, noun_class)
        
        try:
            # Load video
            vr = VideoReader(video_path, num_threads=1, ctx=cpu(0))
            video_fps = vr.get_avg_fps()
            
            # Sample anticipation time
            if self.training:
                at = random.uniform(*self.anticipation_time_sec)
                ap = random.uniform(*self.anticipation_point)
            else:
                at = self.anticipation_time_sec[0]
                ap = self.anticipation_point[0]
            
            # Calculate anticipation frame (frame before action starts)
            aframes = int(at * video_fps)
            af = int(start_frame * ap + (1 - ap) * stop_frame - aframes)
            
            # Calculate frame indices for context
            frame_step = int(video_fps / self.fps)
            num_frames_needed = self.frames_per_clip * frame_step
            
            indices = np.arange(af - num_frames_needed, af, frame_step).astype(np.int64)
            indices = np.clip(indices, 0, len(vr) - 1)
            
            # Load frames
            frames = vr.get_batch(indices).asnumpy()  # (T, H, W, C)
            
            # Transform frames
            transformed_frames = []
            for frame in frames:
                transformed_frames.append(self.transform(frame))
            video_tensor = torch.stack(transformed_frames)  # (T, C, H, W)
            
            # Rearrange to (C, T, H, W) for V-JEPA2
            video_tensor = video_tensor.permute(1, 0, 2, 3)
            
            # Get class indices
            action_class = verb_class * 1000 + noun_class
            
            return {
                'video': video_tensor,
                'verb': self.verb_classes.get(verb_class, 0),
                'noun': self.noun_classes.get(noun_class, 0),
                'action': self.action_classes.get(action_class, 0),
                'anticipation_time': torch.tensor(at, dtype=torch.float32),
                'video_id': video_id,
            }
            
        except Exception as e:
            print(f"Error loading {video_id}: {e}")
            return self._get_dummy_sample(verb_class, noun_class)
    
    def _get_dummy_sample(self, verb_class, noun_class):
        """Return dummy sample when video loading fails"""
        action_class = verb_class * 1000 + noun_class
        return {
            'video': torch.zeros(3, self.frames_per_clip, self.resolution, self.resolution),
            'verb': self.verb_classes.get(verb_class, 0),
            'noun': self.noun_classes.get(noun_class, 0),
            'action': self.action_classes.get(action_class, 0),
            'anticipation_time': torch.tensor(1.0, dtype=torch.float32),
            'video_id': 'dummy',
        }

print("Dataset class defined")

## Load V-JEPA2 Model (Encoder + Predictor)

In [ ]:
# Import V-JEPA2 model components
import src.models.vision_transformer as vit
import src.models.predictor as vit_pred
from src.models.attentive_pooler import AttentivePooler

def load_vjepa2_from_hf():
    """Load V-JEPA2 model from HuggingFace transformers"""
    from transformers import AutoModel
    
    # Load the HuggingFace model
    hf_model = AutoModel.from_pretrained("facebook/vjepa2-vitl-fpc64-256")
    
    return hf_model

def load_vjepa2_from_checkpoint(checkpoint_path: str = None):
    """
    Load V-JEPA2 encoder and predictor from checkpoint.
    If no checkpoint provided, use HuggingFace model.
    """
    if checkpoint_path and Path(checkpoint_path).exists():
        print(f"Loading from checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        
        # Initialize encoder
        encoder = vit.vit_large(
            img_size=CONFIG['resolution'],
            num_frames=CONFIG['frames_per_clip'],
            tubelet_size=CONFIG['encoder']['tubelet_size'],
            patch_size=CONFIG['encoder']['patch_size'],
            uniform_power=CONFIG['encoder']['uniform_power'],
            use_rope=CONFIG['encoder']['use_rope'],
        )
        
        # Load encoder weights
        enc_dict = checkpoint.get('target_encoder', checkpoint.get('encoder', {}))
        enc_dict = {k.replace('module.', '').replace('backbone.', ''): v for k, v in enc_dict.items()}
        encoder.load_state_dict(enc_dict, strict=False)
        
        # Initialize predictor
        predictor = vit_pred.vit_predictor(
            img_size=CONFIG['resolution'],
            embed_dim=encoder.embed_dim,
            patch_size=encoder.patch_size,
            tubelet_size=encoder.tubelet_size,
            num_frames=CONFIG['predictor']['num_frames'],
            depth=CONFIG['predictor']['depth'],
            num_heads=CONFIG['predictor']['num_heads'],
            predictor_embed_dim=CONFIG['predictor']['predictor_embed_dim'],
            num_mask_tokens=CONFIG['predictor']['num_mask_tokens'],
            uniform_power=CONFIG['predictor']['uniform_power'],
            use_mask_tokens=CONFIG['predictor']['use_mask_tokens'],
            use_sdpa=CONFIG['predictor']['use_sdpa'],
            use_rope=CONFIG['predictor']['use_rope'],
        )
        
        # Load predictor weights
        pred_dict = checkpoint.get('predictor', {})
        pred_dict = {k.replace('module.', '').replace('backbone.', ''): v for k, v in pred_dict.items()}
        predictor.load_state_dict(pred_dict, strict=False)
        
        return encoder, predictor, encoder.embed_dim
    else:
        print("Loading V-JEPA2 from HuggingFace...")
        hf_model = load_vjepa2_from_hf()
        return hf_model, None, 1024  # ViT-L embed_dim = 1024

print("V-JEPA2 loading functions defined")

## Attentive Classifier (from V-JEPA2 codebase)

In [ ]:
class AttentiveClassifier(nn.Module):
    """
    Attentive Classifier for action anticipation.
    Uses 3 query tokens for verb, noun, and action prediction.
    Based on V-JEPA2 paper Section 6.
    """

    def __init__(
        self,
        embed_dim: int = 1024,
        num_heads: int = 16,
        depth: int = 4,
        num_verb_classes: int = 97,
        num_noun_classes: int = 300,
        num_action_classes: int = 3806,
    ):
        super().__init__()
        
        self.num_verb_classes = num_verb_classes
        self.action_only = num_verb_classes == 0
        
        # Attentive pooler with 3 queries (verb, noun, action)
        self.pooler = AttentivePooler(
            num_queries=1 if self.action_only else 3,
            embed_dim=embed_dim,
            num_heads=num_heads,
            depth=depth,
            use_activation_checkpointing=True,
        )
        
        # Classifiers
        if not self.action_only:
            self.verb_classifier = nn.Linear(embed_dim, num_verb_classes, bias=True)
            self.noun_classifier = nn.Linear(embed_dim, num_noun_classes, bias=True)
        self.action_classifier = nn.Linear(embed_dim, num_action_classes, bias=True)
        
        self._init_weights()
    
    def _init_weights(self):
        for m in [self.action_classifier]:
            nn.init.normal_(m.weight, std=0.01)
            nn.init.zeros_(m.bias)
        if not self.action_only:
            for m in [self.verb_classifier, self.noun_classifier]:
                nn.init.normal_(m.weight, std=0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Args:
            x: (batch_size, num_tokens, embed_dim) - concatenated encoder + predictor output
        Returns:
            dict with 'verb', 'noun', 'action' logits
        """
        x = self.pooler(x)  # [B, 3, D]
        
        if not self.action_only:
            x_verb, x_noun, x_action = x[:, 0, :], x[:, 1, :], x[:, 2, :]
            return {
                'verb': self.verb_classifier(x_verb),
                'noun': self.noun_classifier(x_noun),
                'action': self.action_classifier(x_action),
            }
        else:
            x_action = x[:, 0, :]
            return {'action': self.action_classifier(x_action)}

print("AttentiveClassifier defined")

## Focal Loss for Class Imbalance

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss for addressing class imbalance (Lin et al., 2017).
    Used in V-JEPA2 for action anticipation.
    """
    
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        p_t = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - p_t) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

print("FocalLoss defined")

Using device: cuda
Probe config: {'embed_dim': 768, 'num_heads': 12, 'num_layers': 4, 'num_action_classes': 3806, 'num_verb_classes': 97, 'num_noun_classes': 300, 'dropout': 0.1}


In [ ]:
## Full Anticipation Model (Frozen Encoder/Predictor + Trainable Probe)

Loaded V-JEPA 2 encoder and predictor (frozen)


In [ ]:
class VJEPA2AnticipationModel(nn.Module):
    """
    Complete V-JEPA2 Action Anticipation Model.
    
    Architecture (from paper):
    1. Frozen encoder processes context video
    2. Frozen predictor predicts future frame representations
    3. Trainable attentive probe classifies action/verb/noun
    """
    
    def __init__(
        self,
        hf_model,
        embed_dim: int = 1024,
        num_heads: int = 16,
        num_probe_blocks: int = 4,
        num_verb_classes: int = 97,
        num_noun_classes: int = 300,
        num_action_classes: int = 3806,
        frames_per_second: int = 4,
        num_output_frames: int = 2,
    ):
        super().__init__()
        
        # Frozen V-JEPA2 model from HuggingFace
        self.vjepa = hf_model
        for param in self.vjepa.parameters():
            param.requires_grad = False
        self.vjepa.eval()
        
        self.embed_dim = embed_dim
        self.frames_per_second = frames_per_second
        self.num_output_frames = num_output_frames
        
        # Grid size for position calculations
        self.grid_size = 256 // 16  # resolution / patch_size = 16
        self.tubelet_size = 2
        
        # Trainable attentive classifier
        self.classifier = AttentiveClassifier(
            embed_dim=embed_dim,
            num_heads=num_heads,
            depth=num_probe_blocks,
            num_verb_classes=num_verb_classes,
            num_noun_classes=num_noun_classes,
            num_action_classes=num_action_classes,
        )
    
    def forward(
        self,
        video: torch.Tensor,
        anticipation_time: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        """
        Args:
            video: (B, C, T, H, W) - video context
            anticipation_time: (B,) - seconds into future to predict
        
        Returns:
            Dict with 'verb', 'noun', 'action' logits
        """
        B = video.shape[0]
        
        with torch.no_grad():
            # Rearrange for HuggingFace model: (B, C, T, H, W) -> (B, T, C, H, W)
            video_input = video.permute(0, 2, 1, 3, 4)
            
            # Get encoder output using full forward pass
            outputs = self.vjepa(video_input, skip_predictor=False)
            
            # Get the combined encoder + predictor features
            # The model concatenates encoder and predictor outputs
            features = outputs.last_hidden_state  # (B, N, D)
        
        # Pass through trainable classifier
        logits = self.classifier(features)
        
        return logits
    
    def train(self, mode: bool = True):
        """Override train to keep vjepa frozen"""
        super().train(mode)
        self.vjepa.eval()  # Always keep frozen
        return self

print("VJEPA2AnticipationModel defined")

Trainable parameters (probe): 41,043,051
Frozen parameters (encoder + predictor): 0


In [ ]:
## Training Loop

Dataset loaders ready (commented out - update paths)


In [ ]:
from tqdm import tqdm

def train_one_epoch(
    model: VJEPA2AnticipationModel,
    train_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion_verb: nn.Module,
    criterion_noun: nn.Module,
    criterion_action: nn.Module,
    device: torch.device,
    epoch: int,
):
    """Train for one epoch"""
    model.train()
    
    total_loss = 0.0
    verb_correct = noun_correct = action_correct = 0
    total_samples = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    for batch in pbar:
        video = batch['video'].to(device)
        verb_labels = batch['verb'].to(device)
        noun_labels = batch['noun'].to(device)
        action_labels = batch['action'].to(device)
        anticipation_time = batch['anticipation_time'].to(device)
        
        # Forward pass
        logits = model(video, anticipation_time)
        
        # Compute losses
        loss_verb = criterion_verb(logits['verb'], verb_labels)
        loss_noun = criterion_noun(logits['noun'], noun_labels)
        loss_action = criterion_action(logits['action'], action_labels)
        loss = loss_verb + loss_noun + loss_action
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.classifier.parameters(), max_norm=1.0)
        optimizer.step()
        
        # Metrics
        total_loss += loss.item()
        verb_correct += (logits['verb'].argmax(1) == verb_labels).sum().item()
        noun_correct += (logits['noun'].argmax(1) == noun_labels).sum().item()
        action_correct += (logits['action'].argmax(1) == action_labels).sum().item()
        total_samples += video.shape[0]
        
        pbar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'verb': f"{100*verb_correct/total_samples:.1f}%",
            'noun': f"{100*noun_correct/total_samples:.1f}%",
        })
    
    return {
        'loss': total_loss / len(train_loader),
        'verb_acc': 100 * verb_correct / total_samples,
        'noun_acc': 100 * noun_correct / total_samples,
        'action_acc': 100 * action_correct / total_samples,
    }


@torch.no_grad()
def validate(
    model: VJEPA2AnticipationModel,
    val_loader: DataLoader,
    device: torch.device,
):
    """Validate the model"""
    model.eval()
    
    verb_correct = noun_correct = action_correct = 0
    total_samples = 0
    
    for batch in tqdm(val_loader, desc="Validation"):
        video = batch['video'].to(device)
        verb_labels = batch['verb'].to(device)
        noun_labels = batch['noun'].to(device)
        action_labels = batch['action'].to(device)
        anticipation_time = batch['anticipation_time'].to(device)
        
        logits = model(video, anticipation_time)
        
        verb_correct += (logits['verb'].argmax(1) == verb_labels).sum().item()
        noun_correct += (logits['noun'].argmax(1) == noun_labels).sum().item()
        action_correct += (logits['action'].argmax(1) == action_labels).sum().item()
        total_samples += video.shape[0]
    
    return {
        'verb_acc': 100 * verb_correct / total_samples,
        'noun_acc': 100 * noun_correct / total_samples,
        'action_acc': 100 * action_correct / total_samples,
    }


def train(
    model: VJEPA2AnticipationModel,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config: Dict,
    device: torch.device,
):
    """Full training loop"""
    
    # Only optimize classifier parameters
    optimizer = torch.optim.AdamW(
        model.classifier.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay'],
    )
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config['num_epochs'],
    )
    
    # Focal loss for each head
    criterion_verb = FocalLoss(alpha=0.25, gamma=2.0)
    criterion_noun = FocalLoss(alpha=0.25, gamma=2.0)
    criterion_action = FocalLoss(alpha=0.25, gamma=2.0)
    
    best_val_acc = 0.0
    save_dir = Path('./checkpoints/ek100_anticipation')
    save_dir.mkdir(parents=True, exist_ok=True)
    
    for epoch in range(1, config['num_epochs'] + 1):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch}/{config['num_epochs']}")
        print(f"{'='*60}")
        
        # Train
        train_metrics = train_one_epoch(
            model, train_loader, optimizer,
            criterion_verb, criterion_noun, criterion_action,
            device, epoch,
        )
        
        # Validate
        val_metrics = validate(model, val_loader, device)
        
        scheduler.step()
        
        # Print metrics
        print(f"\nTrain Loss: {train_metrics['loss']:.4f}")
        print(f"Train - Verb: {train_metrics['verb_acc']:.2f}%, Noun: {train_metrics['noun_acc']:.2f}%, Action: {train_metrics['action_acc']:.2f}%")
        print(f"Val   - Verb: {val_metrics['verb_acc']:.2f}%, Noun: {val_metrics['noun_acc']:.2f}%, Action: {val_metrics['action_acc']:.2f}%")
        
        # Save best model
        if val_metrics['action_acc'] > best_val_acc:
            best_val_acc = val_metrics['action_acc']
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.classifier.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_metrics': val_metrics,
            }, save_dir / 'best_probe.pt')
            print(f"✓ Saved best model (action acc: {best_val_acc:.2f}%)")
    
    return model

print("Training functions defined")

Training setup complete. Uncomment to start training.


## Create Datasets and DataLoaders

In [ ]:
# Create datasets
print("Creating datasets...")

train_dataset = EK100AnticipationDataset(
    annotations=train_df,
    video_root=CONFIG['video_root'],
    verb_to_idx=verb_to_idx,
    noun_to_idx=noun_to_idx,
    action_to_idx=action_to_idx,
    num_frames=CONFIG['num_frames'],
    fps=CONFIG['fps'],
    resolution=CONFIG['resolution'],
    anticipation_time=CONFIG['anticipation_time'],
    augment=True,  # Enable augmentation for training
)

val_dataset = EK100AnticipationDataset(
    annotations=val_df,
    video_root=CONFIG['video_root'],
    verb_to_idx=verb_to_idx,
    noun_to_idx=noun_to_idx,
    action_to_idx=action_to_idx,
    num_frames=CONFIG['num_frames'],
    fps=CONFIG['fps'],
    resolution=CONFIG['resolution'],
    anticipation_time=CONFIG['anticipation_time'],
    augment=False,  # No augmentation for validation
)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True,
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## Initialize Model

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load V-JEPA2 model from HuggingFace
print("\nLoading V-JEPA2 from HuggingFace...")
vjepa2_model, vjepa2_processor = load_vjepa2_from_hf(CONFIG['model_name'])
print(f"Model loaded: {CONFIG['model_name']}")

# Create anticipation model
print("\nCreating anticipation model...")
model = VJEPA2AnticipationModel(
    vjepa2_model=vjepa2_model,
    embed_dim=CONFIG['embed_dim'],
    num_verbs=CONFIG['num_verbs'],
    num_nouns=CONFIG['num_nouns'],
    num_actions=CONFIG['num_actions'],
    num_heads=CONFIG['num_heads'],
    mlp_ratio=CONFIG['mlp_ratio'],
    qkv_bias=CONFIG['qkv_bias'],
    num_queries=CONFIG['num_queries'],
    num_pooler_blocks=CONFIG['num_pooler_blocks'],
)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

## Train!

In [ ]:
# NOTE: Potential issues to debug:
# 1. Video paths: HF cache structure may differ - check if videos exist at expected paths
# 2. Frame extraction: decord may need adjustment for frame indices
# 3. Memory: 64 frames at 256px may require reducing batch_size if OOM
# 4. Model loading: HuggingFace model output format may need adaptation

# Start training
model = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=CONFIG,
    device=device,
)

print("\n" + "="*60)
print("Training complete!")
print("Best checkpoint saved to: ./checkpoints/ek100_anticipation/best_probe.pt")
print("="*60)

## 7. Evaluation Metrics

Additional evaluation metrics for action anticipation.

In [12]:
def compute_metrics(predictions: Dict, targets: Dict) -> Dict:
    """Compute evaluation metrics for action anticipation"""
    
    action_preds = predictions['action']
    verb_preds = predictions['verb']
    noun_preds = predictions['noun']
    
    action_targets = targets['action']
    verb_targets = targets['verb']
    noun_targets = targets['noun']
    
    # Top-1 accuracy
    action_top1 = (action_preds == action_targets).float().mean() * 100
    verb_top1 = (verb_preds == verb_targets).float().mean() * 100
    noun_top1 = (noun_preds == noun_targets).float().mean() * 100
    
    # Combined verb-noun accuracy
    verb_noun_correct = ((verb_preds == verb_targets) & (noun_preds == noun_targets)).float()
    verb_noun_top1 = verb_noun_correct.mean() * 100
    
    return {
        'action_top1': action_top1.item(),
        'verb_top1': verb_top1.item(),
        'noun_top1': noun_top1.item(),
        'verb_noun_top1': verb_noun_top1.item()
    }

print("Evaluation metrics defined")

Evaluation metrics defined


## Notes

### Implementation Details:
1. **Video Sampling**: Clips end exactly 1 second before action start
2. **Future Prediction**: Predictor targets frames 1 second into the future
3. **Frozen Components**: Encoder and predictor weights are not updated
4. **Probe Training**: Only the attentive probe is trainable
5. **Multi-task Learning**: Three independent classifiers with separate focal losses

### TODO:
- [ ] Implement actual V-JEPA 2 encoder/predictor loading
- [ ] Implement video frame loading based on your data format
- [ ] Add data augmentation for video clips
- [ ] Implement top-5 accuracy metrics
- [ ] Add learning rate warmup
- [ ] Add mixed precision training support
- [ ] Update paths to EPIC Kitchen dataset
- [ ] Fine-tune hyperparameters (see Appendix D.1 in paper)